<a href="https://colab.research.google.com/github/gopika-vit/Projects-AI/blob/main/audio_birnn_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 1. IMPORTS
# =========================
import pandas as pd
import numpy as np
import os
import librosa

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# =========================
# 2. LOAD DATA
# =========================
csv_path = "/content/drive/MyDrive/autism_dataset_index.csv"
BASE_PATH = "/content/drive/MyDrive"

df = pd.read_csv(csv_path)

# create full paths
df['audio_path'] = df['voice'].apply(lambda x: os.path.join(BASE_PATH, x))

# =========================
# 3. CLEAN LABELS
# =========================
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].str.strip().str.lower()

label_map = {
    "mild_asd": 0,
    "moderate_asd": 1,
    "severe_asd": 2
}

df['label_num'] = df['label'].map(label_map)
df = df.dropna(subset=['label_num'])
df['label_num'] = df['label_num'].astype(int)

print("Label distribution:")
print(df['label_num'].value_counts())

# =========================
# 4. FEATURE EXTRACTION (STABLE)
# =========================
MAX_LEN = 300
N_MFCC = 40

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)
    y, _ = librosa.effects.trim(y)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)

    # normalize per feature
    mfcc = (mfcc - np.mean(mfcc, axis=1, keepdims=True)) / \
           (np.std(mfcc, axis=1, keepdims=True) + 1e-6)

    mfcc = mfcc.T  # (time, 40)

    # pad / truncate
    if len(mfcc) < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, MAX_LEN - len(mfcc)), (0, 0)))
    else:
        mfcc = mfcc[:MAX_LEN]

    return mfcc

# =========================
# 5. BUILD DATASET
# =========================
X = []
y = []

for i in range(len(df)):
    path = df.iloc[i]['audio_path']
    label = df.iloc[i]['label_num']

    try:
        features = extract_features(path)
        X.append(features)
        y.append(label)
    except Exception as e:
        print("Error:", path, e)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

# sanity check
print("NaN in X:", np.isnan(X).any())
print("NaN in y:", np.isnan(y).any())

# =========================
# 6. TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# 7. ONE-HOT ENCODING
# =========================
num_classes = 3

y_train = to_categorical(y_train_raw, num_classes)
y_test = to_categorical(y_test_raw, num_classes)

# =========================
# 8. CLASS WEIGHTS (VERY IMPORTANT)
# =========================
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
class_weights = dict(enumerate(class_weights))

print("Class weights:", class_weights)

# =========================
# 9. MODEL (SMALL BiLSTM)
# =========================
model = Sequential()

model.add(Bidirectional(LSTM(32, return_sequences=True),
                        input_shape=(300, 40)))
model.add(Dropout(0.3))

model.add(Bidirectional(LSTM(16)))
model.add(Dropout(0.3))

model.add(Dense(32, activation='relu'))
model.add(Dense(3, activation='softmax'))

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =========================
# 10. TRAINING
# =========================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=40,
    batch_size=8,
    class_weight=class_weights,
    callbacks=[early_stop]
)

# =========================
# 11. EVALUATION
# =========================
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

# =========================
# 12. PREDICTIONS (OPTIONAL)
# =========================
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Predictions:", y_pred_classes[:10])
print("Actual:", y_test_raw[:10])

# =========================
# 13. SAVE MODEL
# =========================
model.save("/content/drive/MyDrive/bilstm_autism_model_fixed.h5")


Label distribution:
label_num
2    33
0    23
1    19
Name: count, dtype: int64
X shape: (75, 300, 40)
y shape: (75,)
NaN in X: False
NaN in y: False
Class weights: {0: np.float64(1.0869565217391304), 1: np.float64(1.3157894736842106), 2: np.float64(0.7575757575757576)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_2 (Bidirectional) │ (None, 300, 64)        │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 300, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 32)             │        10,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,211 (118.01 KB)

 Trainable params: 30,211 (118.01 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 126ms/step - accuracy: 0.4333 - loss: 1.0921 - val_accuracy: 0.4000 - val_loss: 1.0874
Epoch 2/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.4000 - loss: 1.0884 - val_accuracy: 0.4000 - val_loss: 1.0861
Epoch 3/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.4333 - loss: 1.0968 - val_accuracy: 0.4000 - val_loss: 1.0842
Epoch 4/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5167 - loss: 1.0789 - val_accuracy: 0.4000 - val_loss: 1.0835
Epoch 5/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5833 - loss: 1.0559 - val_accuracy: 0.4000 - val_loss: 1.0824
Epoch 6/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.6667 - loss: 1.0495 - val_accuracy: 0.4000 - val_loss: 1.0791
Epoch 7/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.6333 - loss: 1.0307 - val_accuracy: 0.3333 - val_loss: 1.0772
Epoch 8/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.6167 - loss: 1.0373 - val_accuracy: 0.4667 - val_loss: 1.0748

Predictions: [2 0 2 2 2 2 2 0 2 2]
Actual: [2 0 0 1 2 2 1 1 1 2]


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/bilstm_autism_model_fixed.h5")

In [ ]:
import librosa
import numpy as np

MAX_LEN = 300
N_MFCC = 40

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)
    y, _ = librosa.effects.trim(y)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)

    mfcc = (mfcc - np.mean(mfcc, axis=1, keepdims=True)) / \
           (np.std(mfcc, axis=1, keepdims=True) + 1e-6)

    mfcc = mfcc.T

    if len(mfcc) < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, MAX_LEN - len(mfcc)), (0, 0)))
    else:
        mfcc = mfcc[:MAX_LEN]

    return mfcc

In [ ]:
file_path = "/content/drive/MyDrive/voice/child_001.wav"

features = extract_features(file_path)

# add batch dimension → (1, 300, 40)
features = np.expand_dims(features, axis=0)

prediction = model.predict(features)

predicted_class = np.argmax(prediction, axis=1)[0]

print("Raw prediction:", prediction)
print("Predicted class:", predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 390ms/step
Raw prediction: [[0.4624084  0.2227953  0.31479633]]
Predicted class: 0


In [ ]:
label_map_reverse = {
    0: "mild_asd",
    1: "moderate_asd",
    2: "severe_asd"
}

print("Final Prediction:", label_map_reverse[predicted_class])

Final Prediction: mild_asd
